In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loaders._gen_binary import generate_data

In [4]:
data = generate_data(seed=0, verbose=True)

Generated dataset with 1995 samples.
Shapes: [(1596, 25), (0, 25), (399, 25)]
Label distribution: Counter({np.int64(1): 817, np.int64(0): 779})
[WARNING] If you use a tree-based model, consider setting use_scaler=False.


In [5]:
X_train, y_train = data['train']
X_test, y_test = data['test']

In [ ]:
# Random Fourier Features (RFF) + Logistic Regression (non-ensemble)
# - One shared RFF mapping (W, b) for all seeds
# - RBF kernel approx: w ~ N(0, 2*gamma I), b ~ Uniform[0, 2π]
# - Evaluate balanced accuracy over 30 seeds

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from loaders._gen_binary import generate_data

# RFF parameters (can be tuned)
D = 500           # projected dimension
gamma = 0.5       # RBF kernel parameter in exp(-gamma * ||x-x'||^2)

W = None
b = None
results = []

for seed in range(30):
    data = generate_data(seed=seed, verbose=False)
    X_train, y_train = data['train']
    X_test, y_test = data['test']

    # Initialize one shared mapping once with training dimensionality
    if W is None:
        d = X_train.shape[1]
        rng = np.random.RandomState(42)
        W = rng.normal(loc=0.0, scale=np.sqrt(2 * gamma), size=(D, d))
        b = rng.uniform(low=0.0, high=2 * np.pi, size=D)

    # Feature map: phi(x) = sqrt(2/D) * cos(Wx + b)
    def rff_transform(X):
        return np.sqrt(2.0 / D) * np.cos(X @ W.T + b)

    Phi_tr = rff_transform(X_train)
    Phi_te = rff_transform(X_test)

    # Linear model on phi(x)
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(Phi_tr, y_train)
    y_pred = lr.predict(Phi_te)

    results.append(balanced_accuracy_score(y_test, y_pred))

results = np.array(results)
print(f"RFF + LogisticRegression over 30 seeds -> Mean: {results.mean():.4f}, Std: {results.std():.4f}")
